## Implementacion de SVDpp para abordar el problema como si se tratase de filtrado colaborativo

Realizamos imports y cargamos los datasets

In [2]:
import numpy as np
import pandas as pd

SEED = 42

In [3]:
business_data = pd.read_csv("./data/negocios.csv")
user_data = pd.read_csv("./data/usuarios.csv")

train_reviews = pd.read_csv("./data/train_reviews.csv")
test_reviews = pd.read_csv("./data/test_reviews.csv")


C:\Users\pable\AppData\Local\Temp\ipykernel_2168\824715385.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  user_data = pd.read_csv("./data/usuarios.csv")


Vamos a intentar resolver esta tarea mediante un enfoque de filtrado colaborativo.

Observamos si existen usuarios o items que no esten en train pero si en test

In [4]:
user_train_set = set(train_reviews['user_id'].unique())
user_test_set = set(test_reviews['user_id'].unique())
print(f"Numero de usuarios en train: {len(user_train_set)}")
print(f"Numero de usuarios en test: {len(user_test_set)}")
non_intersecting_users = user_test_set - user_train_set
print(f"Numero de usuarios en test que no estan en train: {len(non_intersecting_users)}")

item_train_set = set(train_reviews['business_id'].unique())
item_test_set = set(test_reviews['business_id'].unique())
print(f"Numero de items en train: {len(item_train_set)}")
print(f"Numero de items en test: {len(item_test_set)}")
non_intersecting_items = item_test_set - item_train_set
print(f"Numero de items en test que no estan en train: {len(non_intersecting_items)}")

Numero de usuarios en train: 541915
Numero de usuarios en test: 282100
Numero de usuarios en test que no estan en train: 157706
Numero de items en train: 30064
Numero de items en test: 28913
Numero de items en test que no estan en train: 5


Mas de la mitad de los usuarios de test no estan en train (de 282100 de test 157706 faltan, un  56%), esto supone una gran desventaja para el filtrado colavorativo, ya que para funcionar de manera optima, necesita que todos los usuarios e items aparezcan al menos una vez en train. Sin embargo vamos a ver que resultados se obtienen mediante este enfoque.

In [4]:
import pandas as pd
from surprise import Dataset, Reader, SVDpp
from surprise.model_selection import train_test_split, cross_validate, GridSearchCV
import numpy as np
import matplotlib.pyplot as plt
from surprise.accuracy import mae

# Step 1: Load and prepare the dataset
def load_data(file_path):
    # Load the data
    df = pd.read_csv(file_path)

    # Define the format of the data
    reader = Reader(rating_scale=(df['stars'].min(), df['stars'].max()))

    # Load the data into the Surprise format
    data = Dataset.load_from_df(df[['user_id', 'business_id', 'stars']], reader)

    return data

# Step 2: Train SVD++ model with hyperparameter tuning
def train_svdpp_model(data):

    print("Performing hyperparameter tuning...")
    # Define parameter grid
    param_grid = {
        'n_factors': [20, 30, 40],
        'n_epochs': [20,25],
        'lr_all': [0.007, 0.01],
        'reg_all': [0.02,0.1]
    
    }

    # Perform grid search
    gs = GridSearchCV(SVDpp, param_grid, measures=['mae'], cv=2)
    gs.fit(data)

    # Get the best parameters
    best_params = gs.best_params['mae']
    print(f"Best parameters: {best_params}")

    # Train with best parameters
    algo = SVDpp(
        n_factors=best_params['n_factors'],
        n_epochs=best_params['n_epochs'],
        lr_all=best_params['lr_all'],
        reg_all=best_params['reg_all'],
        random_state=42,
        verbose=True
    )

    return algo


# Main function to run the whole process
def main(file_path):
    print("Loading data...")
    data = load_data(file_path)

    print("Training SVD++ model...")
    algo = train_svdpp_model(data)

    return algo

# Usage example

# Replace with your actual file path
file_path = "./data/train_reviews.csv"

# Set to False if you want to skip hyperparameter tuning (faster)
algo = main(file_path)

Loading data...
Training SVD++ model...
Performing hyperparameter tuning...
Best parameters: {'n_factors': 20, 'n_epochs': 20, 'lr_all': 0.01, 'reg_all': 0.1}


Finalmente realizamos las predicciones en el test de kaggle, donde sustituimos los usuarios y negocios no vistos por la media.

In [7]:
import pandas as pd
from surprise import Dataset, Reader, SVDpp

# Retrain the model
# Load the data
df = pd.read_csv("./data/train_reviews.csv")
# Define the format of the data
reader = Reader(rating_scale=(df['stars'].min(), df['stars'].max()))
# Load the data into the Surprise format
data = Dataset.load_from_df(df[['user_id', 'business_id', 'stars']], reader)

algo = SVDpp(n_factors=20, n_epochs=20, lr_all=0.01, reg_all=0.1)
algo.fit(data.build_full_trainset())

# Pre-compute necessary sets and means
user_train_set = set(train_reviews['user_id'])
item_train_set = set(train_reviews['business_id'])
global_mean = train_reviews['stars'].mean()

# Pre-compute means for users and items
item_means = train_reviews.groupby('business_id')['stars'].mean().to_dict()
user_means = train_reviews.groupby('user_id')['stars'].mean().to_dict()

predictions = []

for idx, row in enumerate(test_reviews.itertuples(index=False)):
    if idx % 100000 == 0:
        print(f"iter {idx}")

    user_id = row.user_id
    item_id = row.business_id
    review_id = row.review_id

    if user_id not in user_train_set and item_id not in item_train_set:
        predicted_value = global_mean
    elif user_id not in user_train_set:
        predicted_value = item_means.get(item_id, global_mean)
    elif item_id not in item_train_set:
        predicted_value = user_means.get(user_id, global_mean)
    else:
        predicted_value = algo.predict(uid=user_id, iid=item_id).est

    predictions.append({'review_id': review_id, 'stars': predicted_value})

# Export predictions
pd.DataFrame(predictions).to_csv('./SVDpp_mean_unk.csv', index=False)


iter 0
iter 100000
iter 200000
iter 300000
iter 400000


Se obtienen unos resultados de kaggle de 1.02, lo cual es bastante bajo. Esto sugiere que utilizar un enfoque basado en filtrado colaborativo no es la mejor idea para resolver este problema.

Esto puede deberse a que la naturaleza del problema es compleja y se necesitan mas datos que los que usa SVDpp, o a que existen numerosos usuarios que estan en test y no aparecen en train, que simplemente son sustituidos por la media y no usan el algoritmo de SVDpp.